# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# ML-04 — Load FlyRank Warehouse

import duckdb
import os
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect to DuckDB
con = duckdb.connect()

# Authenticate with Hugging Face
con.execute(f"""
CREATE SECRET (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

# FlyRank warehouse
rel = "hf://datasets/FlyRank/internship-warehouse"

# Test access to the main fact table
result = con.sql(f"""
SELECT COUNT(*) AS total_rows
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
""")

print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────┐
│ total_rows │
│   int64    │
├────────────┤
│   78835655 │
└────────────┘



## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Baseline rule

I will rank pages using two observable February 2026 signals:

- **Staleness:** `days_since_last_update`
- **Visibility:** `impressions_feb`

The baseline score is:

**Baseline Action Score = 60% staleness + 40% visibility**

- Staleness is capped at 365 days and contributes up to 60 points.
- Visibility uses a log-scaled February impression count so extremely large pages do not dominate the score.
- The score is used only to prioritize pages for human review.
- It is not a prediction that a refresh will recover traffic and does not claim causality.

### Reason codes

Each page receives exactly one reason code:

- `stale_visible_page` — page is at least 180 days old and has at least 500 February impressions.
- `aging_visible_page` — page is at least 90 days old and has at least 500 February impressions.
- `visible_page` — page has at least 500 February impressions but does not meet the stronger staleness conditions.
- `monitor` — lower-priority page under this baseline rule.

### Action labels

Each page receives one action label based on its score:

- `REFRESH_REVIEW` — highest-priority review candidate.
- `CONTENT_REVIEW` — medium-priority review candidate.
- `MONITOR` — lower-priority monitoring candidate.
- `LOW_PRIORITY` — lowest priority under this rule.

This rule is intentionally transparent and is a review-prioritization heuristic rather than a causal or predictive model.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
import pandas as pd

# ============================================================
# SECTION 2 — BUILD THE RANKED QUEUE
# Step 2: Build the complete February decision frame
# ============================================================

# Exact February 2026 warehouse file
FEB_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-02/data_0.parquet"
)

DIM_CONTENT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/dim_content.parquet"
)

# Aggregate February daily records into one row
# per client + content pair.
#
# IMPORTANT:
# days_since_last_update needs to be calculated by joining with dim_content.

feb = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,

        SUM(COALESCE(f.gsc_impressions, 0))
            FILTER (WHERE f.gsc_data_available = TRUE)
            AS impressions_feb,

        SUM(COALESCE(f.gsc_clicks, 0))
            FILTER (WHERE f.gsc_data_available = TRUE)
            AS clicks_feb,

        COUNT(*)
            FILTER (WHERE f.gsc_data_available = TRUE)
            AS measured_days_feb,

        -- Calculate days_since_last_update by joining with dim_content
        MAX(datediff('day', d.content_created_date, f.report_date))
            AS days_since_last_update

    FROM read_parquet('{FEB_PATH}') AS f
    INNER JOIN read_parquet('{DIM_CONTENT_PATH}') AS d
        ON f.content_hash_id = d.content_hash_id

    GROUP BY
        f.client_hash_id,
        f.content_hash_id
""").df()

# Keep only page/client pairs with measured GSC data
feb = feb[feb["measured_days_feb"] > 0].copy()

# Clean numeric fields
feb["impressions_feb"] = pd.to_numeric(
    feb["impressions_feb"], errors="coerce"
).fillna(0)

feb["clicks_feb"] = pd.to_numeric(
    feb["clicks_feb"], errors="coerce"
).fillna(0)

feb["days_since_last_update"] = pd.to_numeric(
    feb["days_since_last_update"], errors="coerce"
)

# Remove rows where staleness is unavailable
feb = feb.dropna(
    subset=["days_since_last_update"]
).copy()

print("February decision frame:", len(feb))
print(
    "Median February impressions:",
    f"{feb['impressions_feb'].median():,.0f}"
)
print(
    "Median days since last update:",
    f"{feb['days_since_last_update'].median():.0f}"
)

display(feb.head(10))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

February decision frame: 153559
Median February impressions: 119
Median days since last update: 183


,client_hash_id,content_hash_id,impressions_feb,clicks_feb,measured_days_feb,days_since_last_update
18,client_3ffa76342f366962,content_c2eed5ce37894647,2.0,0.0,2,165
20,client_3ffa76342f366962,content_36bee0a093d0711d,3.0,0.0,2,157
46,client_3ffa76342f366962,content_04bed7e232ca42cd,1.0,0.0,1,163
47,client_3ffa76342f366962,content_0239c0a89e9454c5,1.0,0.0,1,156
53,client_3ffa76342f366962,content_4e2d90da35481342,3.0,0.0,3,173
57,client_3ffa76342f366962,content_cfbf60a6d1558926,1.0,0.0,1,169
80,client_3ffa76342f366962,content_07ffb120aa05fbb6,2.0,0.0,2,170
82,client_3ffa76342f366962,content_1546aabff77c05a4,5.0,1.0,4,174
86,client_3ffa76342f366962,content_8c28319872b3c87c,1.0,0.0,1,163
88,client_3ffa76342f366962,content_78fce5fb92fdf949,15.0,0.0,9,172


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
import numpy as np
import pandas as pd

# ============================================================
# SECTION 2 — BUILD THE RANKED QUEUE (Re-added from deleted cell)
# ============================================================

# Use the February decision frame already created in Section 0/1.
work = feb.copy()

# ------------------------------------------------------------
# 1. Staleness component — 60%
# ------------------------------------------------------------

work["staleness_component"] = (
    np.clip(
        work["days_since_last_update"] / 365.0,
        0,
        1
    ) * 60
)

# ------------------------------------------------------------
# 2. Visibility component — 40%
# ------------------------------------------------------------

work["visibility_component"] = (
    np.clip(
        np.log1p(work["impressions_feb"]) /
        np.log1p(100000),
        0,
        1
    ) * 40
)

# ------------------------------------------------------------
# 3. Final baseline action score
# ------------------------------------------------------------

work["baseline_action_score"] = (
    work["staleness_component"]
    + work["visibility_component"]
)

# ------------------------------------------------------------
# 4. Reason codes
# ------------------------------------------------------------

work["reason_code"] = np.select(
    [
        (
            (work["days_since_last_update"] >= 180)
            & (work["impressions_feb"] >= 500)
        ),

        (
            (work["days_since_last_update"] >= 90)
            & (work["impressions_feb"] >= 500)
        ),

        work["impressions_feb"] >= 500
    ],

    [
        "stale_visible_page",
        "aging_visible_page",
        "visible_page"
    ],

    default="monitor"
)

# ------------------------------------------------------------
# 5. Action labels
# ------------------------------------------------------------

work["action_label"] = np.select(
    [
        work["baseline_action_score"] >= 70,
        work["baseline_action_score"] >= 50,
        work["baseline_action_score"] >= 25
    ],

    [
        "REFRESH_REVIEW",
        "CONTENT_REVIEW",
        "MONITOR"
    ],

    default="LOW_PRIORITY"
)

# ------------------------------------------------------------
# 6. Select queue columns
# ------------------------------------------------------------

queue_cols = [
    "client_hash_id",
    "content_hash_id",
    "baseline_action_score",
    "reason_code",
    "action_label",
    "days_since_last_update",
    "impressions_feb",
    "measured_days_feb"
]

# ------------------------------------------------------------
# 7. Rank the pages
# ------------------------------------------------------------

queue = (
    work.sort_values(
        [
            "baseline_action_score",
            "impressions_feb",
            "content_hash_id"
        ],
        ascending=[False, False, True]
    )[queue_cols]
    .reset_index(drop=True)
)

# Rank starts from 1
queue.index = queue.index + 1
queue.index.name = "rank"

# ============================================================
# SECTION 3 — TOP-20 SKEPTICAL REVIEW
# ============================================================

# Take the 20 highest-ranked pages
top20 = queue.head(20).copy()

# Convert 'rank' index to a column
top20 = top20.reset_index()

# Explain why each page appears in the review
top20["why_it_is_here"] = (
    top20["reason_code"].map({
        "stale_visible_page":
            "Old/stale content with meaningful current visibility.",
        "aging_visible_page":
            "Aging content with meaningful current visibility.",
        "visible_page":
            "Meaningfully visible page with lower staleness risk.",
        "monitor":
            "Lower current-window priority under this rule."
    })
    .fillna("Rule-based monitoring candidate.")
)

# Explicitly state what could make the recommendation wrong
top20["what_would_make_it_wrong"] = (
    "The traffic may be temporary/noisy, the page may still satisfy "
    "the search need, another page may better satisfy the same need, "
    "or the content may not be the right business priority. "
    "The score does not establish that a refresh will improve performance."
)

# Select the final review columns
review = top20[
    [
        "rank",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "action_label",
        "why_it_is_here",
        "what_would_make_it_wrong"
    ]
]

# Display the skeptical review
print("TOP-20 SKEPTICAL REVIEW")
print("=" * 100)

display(review)

TOP-20 SKEPTICAL REVIEW


,rank,content_hash_id,baseline_action_score,reason_code,action_label,why_it_is_here,what_would_make_it_wrong
0,1,content_8e1334d6356668e3,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
1,2,content_fec55986a1868d62,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
2,3,content_e241d6415ac9e534,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
3,4,content_00d4fdf6e48a2d38,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
4,5,content_c9f840183215651b,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
5,6,content_cf651123f1085418,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
6,7,content_ec2e0346994fb5a5,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
7,8,content_fd2117c2c6790e4b,100.000000,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
8,9,content_b17c1d1cb0a346d6,99.401224,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."
9,10,content_d508c9c6173af446,99.103109,stale_visible_page,REFRESH_REVIEW,Old/stale content with meaningful current visi...,"The traffic may be temporary/noisy, the page m..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [4]:
# ============================================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ============================================================

# Show the 5 lowest-ranked pages
weak = queue.tail(5).reset_index()[
    [
        "rank",
        "content_hash_id",
        "baseline_action_score",
        "reason_code",
        "action_label",
        "days_since_last_update",
        "impressions_feb"
    ]
]

print("Weak / low-priority examples:")
print("=" * 100)
display(weak)


# ------------------------------------------------------------
# Check exactly which fields are used for scoring
# ------------------------------------------------------------

scored_inputs = {
    "days_since_last_update",
    "impressions_feb"
}

print("\nScoring inputs:")
print(sorted(scored_inputs))


# ------------------------------------------------------------
# Check for future or label-derived fields
# ------------------------------------------------------------

future_or_label_names = {
    "trend_direction",
    "trend_pct",
    "is_declining",
    "health_score",
    "priority_score",
    "action_type",
    "refresh_tier",
    "imp_mar",
    "clk_mar",
    "march_impressions",
    "future_impressions"
}

used_names = set(work.columns)

leak_names_found = sorted(
    used_names & future_or_label_names
)

print("\nLeakage check:")
print("Future/label fields found in working frame:")
print(leak_names_found)


# ------------------------------------------------------------
# Assertions
# ------------------------------------------------------------

assert not leak_names_found, (
    f"Potential leakage fields: {leak_names_found}"
)

assert {
    "days_since_last_update",
    "impressions_feb"
}.issubset(work.columns)

print("\nPASS — baseline uses current-window observable signals only.")

Weak / low-priority examples:


,rank,content_hash_id,baseline_action_score,reason_code,action_label,days_since_last_update,impressions_feb
0,153555,content_78bc1477c1270d98,2.408238,monitor,LOW_PRIORITY,0,1.0
1,153556,content_810252900ac28e30,2.408238,monitor,LOW_PRIORITY,0,1.0
2,153557,content_9423f1972f4757d5,2.408238,monitor,LOW_PRIORITY,0,1.0
3,153558,content_b8912aaed9de5bcf,2.408238,monitor,LOW_PRIORITY,0,1.0
4,153559,content_c74ffaa4dbb35033,2.408238,monitor,LOW_PRIORITY,0,1.0



Scoring inputs:
['days_since_last_update', 'impressions_feb']

Leakage check:
Future/label fields found in working frame:
[]

PASS — baseline uses current-window observable signals only.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.